# Blog Search Agent — Mode Testing

Interactive notebook to test both modes of the blog search agent:

1. **Chat mode** — conversational, free-form natural language, multi-turn capable
2. **Workflow mode** — deterministic pipeline, structured JSON output with Pydantic validation

### Prerequisites
- AWS credentials configured (`AWS_PROFILE` or env vars)
- `.env` file with `GATEWAY_URL`, `COGNITO_*` vars (or `GATEWAY_TOKEN`)
- `uv sync` to install dependencies

In [1]:
import sys
import os
import json
import asyncio
import logging
from pathlib import Path

# Ensure blog_search is importable
sys.path.insert(0, str(Path.cwd()))

# Suppress noisy loggers
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")
for noisy in ["botocore", "urllib3", "httpcore", "httpx", "mcp.client",
              "strands.tools.mcp", "strands.models", "strands.agent.agent_executor",
              "strands.agent.event_loop", "strands.telemetry"]:
    logging.getLogger(noisy).setLevel(logging.WARNING)

In [2]:
from blog_search_agent import (
    create_chat_agent,
    run_blog_search_workflow,
    extract_json_from_response,
)
from models import EVENT_TYPES, validate_response

print(f"Event types: {EVENT_TYPES}")

Event types: ['INJURY', 'CANCELLATION', 'VENUE_CHANGE', 'SCHEDULE_CHANGE', 'ROSTER']


---
## 1. Chat Mode

Chat mode is conversational — it accepts free-form questions and responds with cited, grounded answers.
The agent has access to: `registry_lookup`, `rss_fetch`, `web_fetch`, `gateway__WebSearch`.

### 1.1 Basic event search

In [3]:
# Create a chat agent (no memory — local testing without AgentCore session)
chat_agent = create_chat_agent(debug=True)

# Ask about recent events for a team
response = chat_agent("Find any recent news for Colgate Raiders basketball")
print("\n" + "="*70)
print("FINAL RESPONSE:")
print("="*70)
print(str(response))


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  SYSTEM PROMPT [CHAT MODE]
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  │ You are an expert sports event extraction agent for trading operations. You search college fan blogs and news sources to extract information that matters to traders — injury news, game schedule changes, venue updates, roster moves, and cancellations.
  │ 
  │ Respond conversationally and concisely. Always ground your answers in information extracted from sources — never fabricate facts.
  │ 
  │ ---
  │ 
  │ ## Sport Scope
  │ 
  │ You ONLY cover: **NCAA Men's Basketball**
  │ 
  │ Do NOT search for, return, or discuss events from any other sport (including women's basketball, football, baseball, etc.). If the user asks about a different sport, respond politely: "I specialize in NCAA Men's Basketball event extraction. I can help you find injury news, schedule changes, venue updates, roster moves, and cancellations

2026-07-14 18:05:25,001 [INFO] tools.registry_reader: [REGISTRY] team=Colgate Raiders total=6 searchable=4 (accessible + not outdated) with_rss=4


Great — 4 sources with RSS feeds. Let me fetch all of them in parallel now!
  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch



2026-07-14 18:05:27,873 [WARNING] tools.rss_fetch: [RSS_FETCH] url=https://www.raiderstalk.com/feed/ bozo_error=text/html is not an XML media type



  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch



...

The RSS feeds didn't surface any men's basketball-specific news in the target window. Let me check the schedule announcement article (which may include men's basketball) and run a catch-all web search simultaneously.
  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: gateway_web-search-tool___WebSearch


  🔧 TOOL CALL: gateway_web-search-tool___WebSearch


  🔧 TOOL CALL: gateway_web-search-tool___WebSearch


  🔧 TOOL CALL: gateway_web-search-tool___WebSearch


  🔧 TOOL CALL: gateway_web-search-tool___WebSearch


  🔧 TOOL CALL: gateway_web-search-tool___WebSearch


  🔧 TO

2026-07-14 18:05:32,854 [INFO] tools.web_fetch: [WEB_FETCH] url=https://colgateathletics.com/news/2026/7/2/general-colgate-athletics-announces-fall-schedules.aspx content_length=11671 dates=0


I found a relevant roster news article from June 16, 2026. Let me fetch the full article for complete details, and also check the current 2026-27 roster page for any additional moves.
  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TO

2026-07-14 18:05:37,123 [INFO] tools.web_fetch: [WEB_FETCH] url=https://colgateathletics.com/sports/mens-basketball/roster content_length=11952 dates=0
2026-07-14 18:05:37,949 [INFO] tools.web_fetch: [WEB_FETCH] url=https://colgateathletics.com/news/2026/6/15/mens-basketball-mens-basketball-announces-incoming-class.aspx content_length=6722 dates=0


Here's a full summary of what I found for **Colgate Raiders Men's Basketball**:

---

## 🏀 Colgate Raiders Men's Basketball — News Summary

### 📋 ROSTER — Incoming Class Announced *(June 16, 2026)*
Head Coach **Matt Langel** announced **three additions** to the 2026-27 roster. These players are now confirmed on the official roster page.

| Player | Pos | Ht/Wt | Background |
|---|---|---|---|
| **Jayden Harrington** | G | 6-4 / 180 | First-year from New Hampton School (Glen Ridge, Mass.) |
| **Kingston Mckoy** | G | 6-4 / 205 | First-year from Berks Catholic (Reading, Pa.) — 2026 PIAA 4A State Player of the Year |
| **Isaiah Skinn** | G | 6-0 / 170 | Sophomore transfer from George Mason (son of George Mason HC Tony Skinn) |

These three join **11 returners** from a team that went **18-15** and made a **ninth consecutive Patriot League semifinal** appearance in 2025-26.
📎 *Source: [colgateathletics.com](https://colgateathletics.com/news/2026/6/15/mens-basketball-mens-basketball-announce

### 1.2 Follow-up question (same agent instance = same conversation context)

In [4]:
# Follow-up on the same agent — tests multi-turn within a session
followup_response = chat_agent("Can you give me more details on the most recent event you found?")
print("\n" + "="*70)
print("FOLLOW-UP RESPONSE:")
print("="*70)
print(str(followup_response))

The most recent event I found was the **Colgate Athletics Fall Schedules announcement** (July 2, 2026) from the official athletics site — but that didn't cover men's basketball. The most recent **men's basketball-specific** event was the **Incoming Class announcement** (June 16, 2026).

Let me pull up the full article on that — actually, I already fetched it in full during our last search! Here are the complete details:

---

## 🏀 Colgate Men's Basketball — 2026-27 Incoming Class *(June 16, 2026)*

**Head Coach Matt Langel** announced three new additions, all guards, joining the program for 2026-27:

---

### 1. Jayden Harrington — G | 6-4 | 180 lbs | #21
- **From:** Glen Ridge, Mass. / New Hampton School
- **Class:** First-Year
- Four-year varsity letterwinner; transferred to New Hampton from Seton Hall Prep
- Appointed **team captain** for 2025-26 senior year
- Led New Hampton to **back-to-back NEPSAC Finalist appearances**
- Two-time NEPSAC Honorable Mention; Lakes Region All-Star; 

### 1.3 Out-of-scope request

In [5]:
# Out-of-scope: should respond politely and explain what it can help with
oos_agent = create_chat_agent(debug=False)
oos_response = oos_agent("What's the weather like in New York today?")
print(str(oos_response))

I specialize in NCAA Men's Basketball event extraction. I can help you find injury news, schedule changes, venue updates, roster moves, and cancellations for NCAA men's basketball teams.

Would you like me to search for a specific team? For example, I can look up the latest news for teams like the Duke Blue Devils, Kentucky Wildcats, Kansas Jayhawks, or any other NCAA men's basketball program!



### 1.4 Greeting (conversational handling)

In [6]:
# Greeting: should respond conversationally, not attempt tool calls
greet_agent = create_chat_agent(debug=False)
greet_response = greet_agent("Hi! What can you help me with?")
print(str(greet_response))

Hi! I'm a **sports event extraction agent** specialized in **NCAA Men's Basketball**. I help traders stay on top of information that matters for their operations.

Here's what I can do for you:

### 🏀 Event Types I Track
| Event Type | What I Look For |
|---|---|
| 🤕 **Injury** | Player injuries, return timelines, practice status |
| ❌ **Cancellation** | Game cancellations, forfeits |
| 🏟️ **Venue Change** | Arena switches, neutral site changes |
| 📅 **Schedule Change** | Tip-off time changes, date changes |
| 📋 **Roster Move** | Transfers, suspensions, eligibility updates |

### 🔍 How It Works
Just give me a **team name** (e.g., *"Duke Blue Devils"* or *"Kansas Jayhawks"*) and I'll:
1. Look up their fan blogs and news sources
2. Scan RSS feeds and articles
3. Surface any relevant events from the **current search window (July 7–14, 2026)**

### 💡 Example Queries
- *"Any injury news for the Kentucky Wildcats?"*
- *"Are there any schedule changes for Duke?"*
- *"Check all event types for

### 1.5 Specific event type request

In [7]:
# Ask for a specific event type
specific_agent = create_chat_agent(debug=True)
specific_response = specific_agent(
    "Are there any schedule changes or venue changes for Duke Blue Devils in the past 2 weeks?"
)
print("\n" + "="*70)
print("RESPONSE:")
print("="*70)
print(str(specific_response))


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  SYSTEM PROMPT [CHAT MODE]
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  │ You are an expert sports event extraction agent for trading operations. You search college fan blogs and news sources to extract information that matters to traders — injury news, game schedule changes, venue updates, roster moves, and cancellations.
  │ 
  │ Respond conversationally and concisely. Always ground your answers in information extracted from sources — never fabricate facts.
  │ 
  │ ---
  │ 
  │ ## Sport Scope
  │ 
  │ You ONLY cover: **NCAA Men's Basketball**
  │ 
  │ Do NOT search for, return, or discuss events from any other sport (including women's basketball, football, baseball, etc.). If the user asks about a different sport, respond politely: "I specialize in NCAA Men's Basketball event extraction. I can help you find injury news, schedule changes, venue updates, roster moves, and cancellations

2026-07-14 18:06:22,688 [INFO] tools.registry_reader: [REGISTRY] team=Duke Blue Devils total=12 searchable=7 (accessible + not outdated) with_rss=2


Great, I found 2 RSS feeds and several other URLs. Let me fetch the RSS feeds first, and simultaneously run a catch-all web search for schedule/venue changes!
  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: rss_fetch


  🔧 TOOL CALL: gateway_web-search-tool___WebSearch



2026-07-14 18:06:26,175 [INFO] tools.rss_fetch: [RSS_FETCH] url=https://pinestateplaybook.substack.com/feed feed_title=Pine State Playbook entries=20 most_recent=2026-05-13



  🔧 TOOL CALL: gateway_web-search-tool___WebSearch


  🔧 TOOL CALL: gateway_web-search-tool___WebSearch


  🔧 TOOL CALL: gateway_web-search-tool___WebSearch


  🔧 TOOL CALL: gateway_web-search-tool___WebSearch


  🔧 TOOL CALL: gateway_web-search-tool___WebSearch


  🔧 TOOL CALL: gateway_web-search-tool___WebSearch


  🔧 TOOL CALL: gateway_web-search-tool___WebSearch


  🔧 TOOL CALL: gateway_web-search-tool___WebSearch


  🔧 TOOL CALL: gateway_web-search-tool___WebSearch


  🔧 TOOL CALL: gateway_web-search-tool___WebSearch


  🔧 TOOL CALL: gateway_web-search-tool___WebSearch



2026-07-14 18:06:26,332 [INFO] tools.rss_fetch: [RSS_FETCH] url=https://forums.dukebasketballreport.com/index.php?forums/-/index.rss feed_title=Duke Basketball Report entries=20 most_recent=2026-07-14


I found a very relevant venue change article. Let me fetch the full details on the Duke/Michigan venue switch story, as it's the most directly relevant result.
  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch



2026-07-14 18:06:31,357 [INFO] tools.web_fetch: [WEB_FETCH] url=https://www.si.com/college/duke/blue-devils-why-michigan-venue-switch-unique content_length=6547 dates=5
2026-07-14 18:06:31,379 [INFO] tools.web_fetch: [WEB_FETCH] url=https://www.si.com/college/duke/blue-devils-basketball-date-gonzaga-just-took-major-hit content_length=7814 dates=4


Here's a full summary of what I found for Duke Blue Devils NCAA Men's Basketball — covering both **venue changes** and **schedule-related news** within the past two weeks (July 7–14, 2026):

---

## 🏟️ VENUE CHANGE — Duke vs. Michigan (Dec. 21, 2026)

**Event:** Duke Blue Devils vs. Michigan Wolverines
**Original Venue:** Madison Square Garden, New York City
**New Venue:** loanDepot Park, Miami, FL *(home of MLB's Miami Marlins — first basketball game ever hosted there)*
**Date of Game:** December 21, 2026
**Why the Change:** A broadcasting rights dispute triggered the move. Duke had agreed to an Amazon Prime Video deal to broadcast the game, but Fox initially held territorial broadcast rights for MSG. To resolve the conflict, the game was relocated to loanDepot Park in Miami — a highly unconventional choice, as the venue has never hosted a basketball game.
**Published:** June 6, 2026
📰 Source: [Sports Illustrated – "Why Duke/Michigan Venue Switch Is So Unique"](https://www.si.com/coll

---
## 2. Workflow Mode

Workflow mode is a deterministic pipeline for batch/scheduled runs:
1. Fetches team blog URLs from DynamoDB registry
2. Runs a focused agent (RSS + web search) with strict structured output
3. Parses & validates with Pydantic (retries on failure)
4. Filters results by date window

### 2.1 Standard workflow run (today only — default)

In [8]:
result = await run_blog_search_workflow(
    team="Colgate Raiders",
    sport="NCAA Men's Basketball",
    lookback_days=0,  # today only (default for workflow mode)
    debug=True,
)

print("\n" + "="*70)
print("WORKFLOW RESULT:")
print("="*70)
print(json.dumps(result, indent=2, default=str))

2026-07-14 18:06:45,706 [INFO] blog_search_agent: [WORKFLOW] Fetching registry for team=Colgate Raiders



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  INPUT PAYLOAD
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  │ {
  │   "mode": "workflow",
  │   "team": "Colgate Raiders",
  │   "sport": "NCAA Men's Basketball",
  │   "events": [
  │     "INJURY",
  │     "CANCELLATION",
  │     "VENUE_CHANGE",
  │     "SCHEDULE_CHANGE",
  │     "ROSTER"
  │   ],
  │   "lookback_days": 0
  │ }
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━



2026-07-14 18:06:46,068 [INFO] tools.registry_reader: [REGISTRY] team=Colgate Raiders total=6 searchable=4 (accessible + not outdated) with_rss=4
2026-07-14 18:06:46,069 [INFO] blog_search_agent: [WORKFLOW] Running agent: team=Colgate Raiders urls=4 date_range=2026-07-14 to 2026-07-14
2026-07-14 18:06:46,182 [ERROR] blog_search_agent: [WORKFLOW] Agent execution failed for team=Colgate Raiders
Traceback (most recent call last):
  File "/Users/priyalex/Documents/projects/fanduel/ncaa-blog-agent/blog_search/blog_search_agent.py", line 634, in run_blog_search_workflow
    agent = _create_workflow_agent(debug=debug)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/priyalex/Documents/projects/fanduel/ncaa-blog-agent/blog_search/blog_search_agent.py", line 370, in _create_workflow_agent
    system_prompt = load_prompt("workflow_system_prompt").format(
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
KeyError: 'domain'



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  REGISTRY URLs
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  │ Team: Colgate Raiders
  │ Total searchable URLs: 4
  │ 
  │ With RSS (4):
  │   https://colgateathletics.com/sports/mens-basketball
  │     ↳ RSS: https://colgateathletics.com/rss
  │   https://thecolgatemaroonnews.com/category/sports/
  │     ↳ RSS: https://thecolgatemaroonnews.com/feed/
  │   https://thecolgatemaroonnews.com/68770/sports/colgate-mens-basketball-finishes-2025-26-season-with-a-sense-of-optimism/
  │     ↳ RSS: https://thecolgatemaroonnews.com/feed/
  │   https://www.raiderstalk.com/
  │     ↳ RSS: https://www.raiderstalk.com/feed/
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  USER MESSAGE → AGENT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  │ Team: Colgate Raiders
  │ Sport: NCAA Men

### 2.2 Validate the output schema

In [9]:
if result.get("status") == "success":
    # Re-validate the returned data against Pydantic models
    validation = validate_response({
        "results": result["results"],
        "retrieval_diagnostics": result["retrieval_diagnostics"],
    })
    if isinstance(validation, list):
        print("VALIDATION ERRORS:")
        for err in validation:
            print(f"  - {err}")
    else:
        print(f"Validation PASSED: {len(validation.results)} events")
        for ev in validation.results:
            print(f"  [{ev.event_type}] {ev.summary[:80]}")
else:
    print(f"Workflow returned error: {result.get('error')}")

Workflow returned error: Agent execution failed: 'domain'


### 2.3 Workflow with specific event types

In [10]:
# Request only INJURY and ROSTER — INJURY is always force-included
result_filtered = await run_blog_search_workflow(
    team="Duke Blue Devils",
    sport="NCAA Men's Basketball",
    events=["ROSTER"],  # INJURY will be auto-prepended
    lookback_days=0,  # today only
    debug=True,
)

print("\n" + "="*70)
print(f"Status: {result_filtered.get('status')}")
print(f"Results before date filter: {result_filtered.get('results_before_filter')}")
print(f"Results after date filter: {result_filtered.get('results_after_filter')}")
print(f"Date range: {result_filtered.get('date_range')}")
print("="*70)

for ev in result_filtered.get("results", []):
    print(f"  [{ev['event_type']}] {ev['summary'][:80]}")
    print(f"    Source: {ev['source_url'][:60]}")
    print(f"    Date: {ev.get('blog_post_date', 'unknown')}")
    print()

2026-07-14 18:06:46,192 [INFO] blog_search_agent: [WORKFLOW] Fetching registry for team=Duke Blue Devils



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  INPUT PAYLOAD
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  │ {
  │   "mode": "workflow",
  │   "team": "Duke Blue Devils",
  │   "sport": "NCAA Men's Basketball",
  │   "events": [
  │     "INJURY",
  │     "ROSTER"
  │   ],
  │   "lookback_days": 0
  │ }
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━



2026-07-14 18:06:46,435 [INFO] tools.registry_reader: [REGISTRY] team=Duke Blue Devils total=12 searchable=7 (accessible + not outdated) with_rss=2
2026-07-14 18:06:46,436 [INFO] blog_search_agent: [WORKFLOW] Running agent: team=Duke Blue Devils urls=7 date_range=2026-07-14 to 2026-07-14
2026-07-14 18:06:46,566 [ERROR] blog_search_agent: [WORKFLOW] Agent execution failed for team=Duke Blue Devils
Traceback (most recent call last):
  File "/Users/priyalex/Documents/projects/fanduel/ncaa-blog-agent/blog_search/blog_search_agent.py", line 634, in run_blog_search_workflow
    agent = _create_workflow_agent(debug=debug)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/priyalex/Documents/projects/fanduel/ncaa-blog-agent/blog_search/blog_search_agent.py", line 370, in _create_workflow_agent
    system_prompt = load_prompt("workflow_system_prompt").format(
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
KeyError: 'domain'



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  REGISTRY URLs
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  │ Team: Duke Blue Devils
  │ Total searchable URLs: 7
  │ 
  │ With RSS (2):
  │   https://forums.dukebasketballreport.com/index.php
  │     ↳ RSS: https://forums.dukebasketballreport.com/index.php?forums/-/index.rss
  │   https://pinestateplaybook.substack.com/p/its-finally-time-for-me-to-talk-about
  │     ↳ RSS: https://pinestateplaybook.substack.com/feed
  │ 
  │ Without RSS (5):
  │   https://247sports.com/college/duke/board/duke-blue-devils-basketball-message-board-101614/
  │   https://www.on3.com/boards/forums/duke-hoops-open-forum.322/
  │   https://balldurham.com/
  │   https://goduke.com/sports/mens-basketball
  │   https://goduke.com/sports/2025/9/24/2025-26-duke-mens-basketball-media-hub
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

### 2.4 Workflow with no results (today only, unlikely to have events)

In [11]:
# lookback_days=0 means today only — tests the empty-results path
result_empty = await run_blog_search_workflow(
    team="Chicago State Cougars",
    sport="NCAA Men's Basketball",
    lookback_days=0,
    debug=False,
)

print(f"Status: {result_empty.get('status')}")
print(f"Results: {len(result_empty.get('results', []))}")
print(f"Diagnostics: {json.dumps(result_empty.get('retrieval_diagnostics', {}), indent=2)}")

2026-07-14 18:06:46,571 [INFO] blog_search_agent: [WORKFLOW] Fetching registry for team=Chicago State Cougars
2026-07-14 18:06:46,794 [INFO] tools.registry_reader: [REGISTRY] team=Chicago State Cougars total=4 searchable=2 (accessible + not outdated) with_rss=1
2026-07-14 18:06:46,795 [INFO] blog_search_agent: [WORKFLOW] Running agent: team=Chicago State Cougars urls=2 date_range=2026-07-14 to 2026-07-14
2026-07-14 18:06:46,897 [ERROR] blog_search_agent: [WORKFLOW] Agent execution failed for team=Chicago State Cougars
Traceback (most recent call last):
  File "/Users/priyalex/Documents/projects/fanduel/ncaa-blog-agent/blog_search/blog_search_agent.py", line 634, in run_blog_search_workflow
    agent = _create_workflow_agent(debug=debug)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/priyalex/Documents/projects/fanduel/ncaa-blog-agent/blog_search/blog_search_agent.py", line 370, in _create_workflow_agent
    system_prompt = load_prompt("workflow_system_prompt").format(
 

Status: error
Results: 0
Diagnostics: {}


### 2.5 Workflow with non-existent team (error path)

In [12]:
result_err = await run_blog_search_workflow(
    team="Nonexistent University Falcons",
    sport="NCAA Men's Basketball",
    lookback_days=0,  # today only
    debug=False,
)

print(f"Status: {result_err.get('status')}")
print(f"Error: {result_err.get('error', 'N/A')}")

2026-07-14 18:06:46,901 [INFO] blog_search_agent: [WORKFLOW] Fetching registry for team=Nonexistent University Falcons
2026-07-14 18:06:47,134 [WARNING] tools.registry_reader: [REGISTRY] Team not found: Nonexistent University Falcons


Status: error
Error: Team 'Nonexistent University Falcons' not found in registry


---
## 3. Pydantic Validation (Unit Tests)

Test the validation logic directly without calling the agent.

In [13]:
from models import BlogSearchResponse, EventResult, RetrievalDiagnostics

# Valid response
valid_payload = {
    "results": [
        {
            "sport": "NCAA Men's Basketball",
            "team": "Colgate Raiders",
            "event_type": "INJURY",
            "player_name": "John Smith",
            "excerpt": "Smith will miss 4-6 weeks with a knee injury sustained in practice.",
            "summary": "John Smith out 4-6 weeks with knee injury",
            "source_url": "https://colgatefanblog.com/2026/07/smith-injury",
            "blog_post_date": "2026-07-10",
            "detected_at": "2026-07-14T12:00:00Z",
            "retrieval_method": "rss + web_fetch",
        }
    ],
    "retrieval_diagnostics": {
        "urls_total": 3,
        "urls_with_rss": 2,
        "urls_without_rss": 1,
        "web_searches_performed": 1,
        "rss_feeds_fetched": 2,
        "web_fetches_performed": 1,
        "events_found_via_rss": True,
        "events_found_via_web_search": False,
    },
}

result = validate_response(valid_payload)
assert not isinstance(result, list), f"Expected valid, got errors: {result}"
print(f"Valid payload: PASSED ({len(result.results)} events)")
print(f"  event_type: {result.results[0].event_type}")
print(f"  retrieval_method: {result.results[0].retrieval_method}")

Valid payload: PASSED (1 events)
  event_type: INJURY
  retrieval_method: rss + web_fetch


In [14]:
# Invalid response — bad event_type and missing required field
invalid_payload = {
    "results": [
        {
            "sport": "NCAA Men's Basketball",
            "team": "Colgate Raiders",
            "event_type": "TRADE",  # invalid
            "excerpt": "Some text",
            "summary": "Something happened",
            "source_url": "https://example.com",
            # missing: detected_at, retrieval_method
        }
    ],
    "retrieval_diagnostics": {
        "urls_total": 1,
        "urls_with_rss": 0,
        "urls_without_rss": 1,
        "web_searches_performed": 1,
        "rss_feeds_fetched": 0,
        "web_fetches_performed": 0,
    },
}

result = validate_response(invalid_payload)
assert isinstance(result, list), "Expected validation errors"
print(f"Invalid payload: correctly rejected with {len(result)} error(s):")
for err in result:
    print(f"  - {err}")

Invalid payload: correctly rejected with 3 error(s):
  - results -> 0 -> event_type: Value error, event_type must be one of ['INJURY', 'CANCELLATION', 'VENUE_CHANGE', 'SCHEDULE_CHANGE', 'ROSTER'], got 'TRADE'
  - results -> 0 -> detected_at: Field required
  - results -> 0 -> retrieval_method: Field required


In [15]:
# Empty results (valid — no events found is acceptable)
empty_payload = {
    "results": [],
    "retrieval_diagnostics": {
        "urls_total": 2,
        "urls_with_rss": 1,
        "urls_without_rss": 1,
        "web_searches_performed": 1,
        "rss_feeds_fetched": 1,
        "web_fetches_performed": 0,
    },
}

result = validate_response(empty_payload)
assert not isinstance(result, list), f"Expected valid, got: {result}"
print(f"Empty results payload: PASSED (0 events, valid schema)")

Empty results payload: PASSED (0 events, valid schema)


---
## 4. JSON Extraction (Unit Tests)

Test the balanced-brace JSON extraction from agent response text.

In [16]:
# JSON embedded in markdown code block
text_with_codeblock = '''Here are the results:
```json
{"results": [], "retrieval_diagnostics": {"urls_total": 0, "urls_with_rss": 0, "urls_without_rss": 0, "web_searches_performed": 0, "rss_feeds_fetched": 0, "web_fetches_performed": 0}}
```
'''
parsed = extract_json_from_response(text_with_codeblock)
assert parsed is not None
assert parsed["results"] == []
print("Code block extraction: PASSED")

# Raw JSON (no wrapping)
raw_json = '{"results": [{"sport": "Basketball"}], "retrieval_diagnostics": {"urls_total": 1, "urls_with_rss": 0, "urls_without_rss": 1, "web_searches_performed": 0, "rss_feeds_fetched": 0, "web_fetches_performed": 0}}'
parsed = extract_json_from_response(raw_json)
assert parsed is not None
assert len(parsed["results"]) == 1
print("Raw JSON extraction: PASSED")

# JSON with preamble text
text_with_preamble = 'Based on my search, here is the result: {"results": [], "retrieval_diagnostics": {"urls_total": 2, "urls_with_rss": 1, "urls_without_rss": 1, "web_searches_performed": 1, "rss_feeds_fetched": 1, "web_fetches_performed": 0}}'
parsed = extract_json_from_response(text_with_preamble)
assert parsed is not None
assert parsed["retrieval_diagnostics"]["urls_total"] == 2
print("Preamble extraction: PASSED")

# Non-JSON response (conversational)
conversational = "I couldn't find any recent injury news for that team. Would you like me to check another team?"
parsed = extract_json_from_response(conversational)
assert parsed is None
print("Non-JSON response: correctly returned None")

Code block extraction: PASSED
Raw JSON extraction: PASSED
Preamble extraction: PASSED
Non-JSON response: correctly returned None


---
## 5. Summary

| Test | Mode | What it verifies |
|------|------|------------------|
| 1.1 | Chat | Basic event search with tool calls (7-day default lookback) |
| 1.2 | Chat | Follow-up query (multi-turn context) |
| 1.3 | Chat | Out-of-scope handling |
| 1.4 | Chat | Greeting / non-task handling |
| 1.5 | Chat | Specific event type filtering |
| 2.1 | Workflow | Standard pipeline (today only) |
| 2.2 | Workflow | Pydantic output validation |
| 2.3 | Workflow | Specific event types + INJURY auto-include (today only) |
| 2.4 | Workflow | Empty results path (today only) |
| 2.5 | Workflow | Non-existent team error handling |
| 3.x | Validation | Pydantic model unit tests |
| 4.x | Extraction | JSON parsing from agent text |